# MJLab / MuJoCo rendering diagnostic (no training)

Bisects the Colab G4 (sm_120) render segfault in ~3 minutes. Each step runs in
its own subprocess: a crash marks that step FAIL and the ladder continues.
Run all cells top to bottom; screenshot the SUMMARY block.

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi --query-gpu=driver_version,memory.total --format=csv,noheader
!ldconfig -p | grep -iE 'libEGL|libOSMesa' || echo 'no EGL/OSMesa libs listed'

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'mjlab>=1.5.3', 'mediapy'])
from importlib.metadata import version
print('mjlab', version('mjlab'), '| mujoco-warp', version('mujoco-warp'),
      '| warp-lang', version('warp-lang'), '| mujoco', version('mujoco'))

In [ ]:
%%writefile render_probe.py
# One bisection step per invocation: python render_probe.py <step>
# Each step runs in its own process so a segfault only kills that step.
import os, sys
os.environ.setdefault('MUJOCO_GL', sys.argv[2] if len(sys.argv) > 2 else 'egl')
os.environ.setdefault('PYOPENGL_PLATFORM', 'egl')
step = sys.argv[1]
print(f'[probe:{step}] MUJOCO_GL={os.environ["MUJOCO_GL"]}', flush=True)

if step == 'gl_pure':
    # classic mujoco.Renderer on a trivial model -- NO mjlab, NO warp
    import mujoco
    print('[probe] mujoco', mujoco.__version__, flush=True)
    m = mujoco.MjModel.from_xml_string(
        '<mujoco><worldbody><light pos="0 0 3"/><geom type="plane" size="5 5 .1"/>'
        '<body pos="0 0 1"><freejoint/><geom type="box" size=".1 .1 .1"/></body>'
        '</worldbody></mujoco>')
    d = mujoco.MjData(m)
    mujoco.mj_forward(m, d)
    r = mujoco.Renderer(m, height=240, width=320)
    r.update_scene(d)
    px = r.render()
    print('[probe] PURE GL RENDER OK', px.shape, flush=True)

elif step == 'mjlab_sim':
    # mjlab env, NO render_mode: sim side only, tiny
    import warp as wp; wp.init()
    import torch
    from mjlab.tasks.registry import load_env_cfg
    from mjlab.envs.manager_based_rl_env import ManagerBasedRlEnv
    cfg = load_env_cfg('Mjlab-Velocity-Flat-Unitree-Go1')
    cfg.scene.num_envs = 2
    env = ManagerBasedRlEnv(cfg, device='cuda')
    env.reset()
    for _ in range(5):
        env.step(torch.zeros(2, env.action_space.shape[-1], device='cuda'))
    print('[probe] MJLAB SIM OK', flush=True)

elif step == 'mjlab_render_default':
    # mjlab env WITH render_mode, all-default viewer (no overrides)
    import warp as wp; wp.init()
    import torch
    from mjlab.tasks.registry import load_env_cfg
    from mjlab.envs.manager_based_rl_env import ManagerBasedRlEnv
    cfg = load_env_cfg('Mjlab-Velocity-Flat-Unitree-Go1')
    cfg.scene.num_envs = 2
    env = ManagerBasedRlEnv(cfg, device='cuda', render_mode='rgb_array')
    env.reset()
    f = env.render()
    print('[probe] MJLAB RENDER DEFAULT OK', getattr(f, 'shape', None), flush=True)

elif step == 'mjlab_render_full':
    # mjlab env with OUR viewer overrides (resolution/fovy/light/texrepeat)
    import warp as wp; wp.init()
    import torch
    from dataclasses import replace as _r
    from mjlab.tasks.registry import load_env_cfg
    from mjlab.envs.manager_based_rl_env import ManagerBasedRlEnv
    cfg = load_env_cfg('Mjlab-Velocity-Flat-Unitree-Go1')
    cfg.scene.num_envs = 3
    v = cfg.viewer
    v.width, v.height = 1280, 720
    v.distance, v.elevation, v.azimuth, v.fovy = 1.8, -25.0, 90.0, 32.0
    v.max_extra_envs = 0
    L = cfg.scene.terrain.lights[0]
    d = (-0.4, -0.25, -0.88); n = sum(x*x for x in d) ** 0.5
    try: L.dir = tuple(x/n for x in d)
    except Exception: cfg.scene.terrain.lights = (_r(L, dir=tuple(x/n for x in d)),)
    m = cfg.scene.terrain.materials[0]
    try: m.texrepeat = (2.0, 2.0)
    except Exception: cfg.scene.terrain.materials = (_r(m, texrepeat=(2.0, 2.0)),)
    m = cfg.scene.terrain.materials[0]
    try: m.texrepeat = (2.0, 2.0)
    except Exception: cfg.scene.terrain.materials = (_r(m, texrepeat=(2.0, 2.0)),)
    env = ManagerBasedRlEnv(cfg, device='cuda', render_mode='rgb_array')
    env.reset()
    for _ in range(5):
        env.step(torch.zeros(3, env.action_space.shape[-1], device='cuda'))
        f = env.render()
    print('[probe] MJLAB RENDER FULL OK', f.shape, flush=True)

elif step == 'order_cuda_then_gl':
    # CUDA/warp first, THEN create the EGL context -- mjlab's implicit order
    import warp as wp; wp.init()
    import torch; torch.zeros(8, device='cuda')
    import mujoco
    m = mujoco.MjModel.from_xml_string(
        '<mujoco><worldbody><light pos="0 0 3"/><geom type="plane" size="5 5 .1"/>'
        '</worldbody></mujoco>')
    d = mujoco.MjData(m); mujoco.mj_forward(m, d)
    r = mujoco.Renderer(m, height=240, width=320)
    r.update_scene(d); r.render()
    print('[probe] CUDA->GL ORDER OK', flush=True)

elif step == 'order_gl_then_cuda':
    # EGL context FIRST, then warp/CUDA -- the candidate workaround order
    import mujoco
    m = mujoco.MjModel.from_xml_string(
        '<mujoco><worldbody><light pos="0 0 3"/><geom type="plane" size="5 5 .1"/>'
        '</worldbody></mujoco>')
    d = mujoco.MjData(m); mujoco.mj_forward(m, d)
    r = mujoco.Renderer(m, height=240, width=320)
    r.update_scene(d); r.render()
    import warp as wp; wp.init()
    import torch; torch.zeros(8, device='cuda')
    r.update_scene(d); r.render()
    print('[probe] GL->CUDA ORDER OK', flush=True)

elif step == 'mjlab_render_warmup':
    # full mjlab render, but an EGL context is created BEFORE warp init
    import mujoco
    _wm = mujoco.MjModel.from_xml_string('<mujoco><worldbody/></mujoco>')
    _warm = mujoco.Renderer(_wm, height=64, width=64)   # EGL first
    import warp as wp; wp.init()
    import torch
    from mjlab.tasks.registry import load_env_cfg
    from mjlab.envs.manager_based_rl_env import ManagerBasedRlEnv
    cfg = load_env_cfg('Mjlab-Velocity-Flat-Unitree-Go1')
    cfg.scene.num_envs = 2
    env = ManagerBasedRlEnv(cfg, device='cuda', render_mode='rgb_array')
    env.reset()
    f = env.render()
    print('[probe] MJLAB RENDER WITH GL WARMUP OK', getattr(f, 'shape', None), flush=True)

elif step == 'render_video':
    # the payoff: an ACTUAL rendered video (no policy needed) -- stance under
    # PD control with the env's push events, then a sinusoidal sway.
    import mujoco as _mj
    _warm = _mj.Renderer(_mj.MjModel.from_xml_string('<mujoco><worldbody/></mujoco>'),
                         height=64, width=64)   # GL context BEFORE warp (workaround)
    import warp as wp; wp.init()
    import numpy as np, torch
    import mediapy as media
    from dataclasses import replace as _r
    from mjlab.tasks.registry import load_env_cfg
    from mjlab.envs.manager_based_rl_env import ManagerBasedRlEnv
    cfg = load_env_cfg('Mjlab-Velocity-Flat-Unitree-Go1')
    cfg.scene.num_envs = 2
    v = cfg.viewer
    v.width, v.height = 1280, 720
    v.distance, v.elevation, v.azimuth, v.fovy = 1.8, -25.0, 90.0, 32.0
    v.max_extra_envs = 0
    L = cfg.scene.terrain.lights[0]
    d = (-0.4, -0.25, -0.88); n = sum(x*x for x in d) ** 0.5
    try: L.dir = tuple(x/n for x in d)
    except Exception: cfg.scene.terrain.lights = (_r(L, dir=tuple(x/n for x in d)),)
    m = cfg.scene.terrain.materials[0]
    try: m.texrepeat = (2.0, 2.0)
    except Exception: cfg.scene.terrain.materials = (_r(m, texrepeat=(2.0, 2.0)),)
    env = ManagerBasedRlEnv(cfg, device='cuda', render_mode='rgb_array')
    env.update_visualizers = None
    env.reset()
    nact = env.action_space.shape[-1]
    frames = []
    for t in range(240):
        if t < 120:
            a = torch.zeros(2, nact, device='cuda')          # stance + env pushes
        else:
            phase = 2 * 3.14159 * (t - 120) / 40.0            # slow sway
            a = 0.25 * torch.sin(torch.tensor(phase)) * torch.ones(2, nact, device='cuda')
        env.step(a)
        frames.append(np.asarray(env.render()))
    media.write_video('probe_video.mp4', frames, fps=30)
    import os
    print('[probe] VIDEO OK probe_video.mp4', os.path.getsize('probe_video.mp4') // 1024, 'KiB', flush=True)

else:
    raise SystemExit(f'unknown step {step}')

In [ ]:
import subprocess, sys
STEPS = [('gl_pure', 'egl'),      # mujoco.Renderer alone, EGL
         ('gl_pure', 'osmesa'),   # mujoco.Renderer alone, software GL (if available)
         ('mjlab_sim', 'egl'),    # mjlab sim only, no rendering
         ('mjlab_render_default', 'egl'),
         ('mjlab_render_full', 'egl'),
         ('order_cuda_then_gl', 'egl'),   # mjlab's implicit order, minimal repro
         ('order_gl_then_cuda', 'egl'),   # candidate workaround order
         ('mjlab_render_warmup', 'egl'),  # full mjlab render + workaround
         ('render_video', 'egl')]         # the payoff: an actual video
results = []
for step, gl in STEPS:
    p = subprocess.run([sys.executable, 'render_probe.py', step, gl],
                       capture_output=True, text=True, timeout=1200)
    ok = p.returncode == 0
    results.append((step, gl, p.returncode))
    print('=' * 60)
    print(f'STEP {step} (MUJOCO_GL={gl}):', 'PASS' if ok else f'FAIL exit {p.returncode}')
    print(p.stdout[-1500:])
    if not ok:
        print(p.stderr[-1500:])
print('=' * 60)
print('SUMMARY:')
for step, gl, rc in results:
    print(f'  {step:22s} [{gl:6s}] ->', 'PASS' if rc == 0 else f'FAIL ({rc})')
print("Interpretation: order_cuda_then_gl FAIL + order_gl_then_cuda PASS ="
      " CUDA-before-EGL context conflict (5-line upstream repro; workaround ="
      " create the GL context before warp init). mjlab_render_warmup PASS ="
      " workaround confirmed end-to-end.")

In [ ]:
# The actual video (if the render_video step passed):
import os
if os.path.exists('probe_video.mp4'):
    import mediapy as media
    media.show_video(media.read_video('probe_video.mp4'), fps=30)
else:
    print('no video: render_video step failed -- see the SUMMARY above')